This notebook was executed in Kaggle to get a finetuned FRCNN FPN model based on the CrowdHuman dataset.

In [1]:
#!pip install -U ultralytics
#import ultralytics
#ultralytics.checks()

Ultralytics 8.3.253 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6636.9/8062.4 GB disk)


In [1]:
import gc
import torch
import os

# Limpiar memoria de Python y CUDA
gc.collect()
torch.cuda.empty_cache()

# Configuración sugerida por tu error para evitar fragmentación
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import os
import json
import torch
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import v2 as T
from huggingface_hub import hf_hub_download

# --- CONFIGURACIÓN DE RUTAS ---
BASE_DIR = "/kaggle/working"
SAVE_DIR = os.path.join(BASE_DIR, "models")
DATA_RAW = os.path.join(BASE_DIR, "dataset_raw")

# Carpetas finales para el entrenamiento
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "CrowdHuman_train/Images")
VAL_IMG_DIR = os.path.join(BASE_DIR, "CrowdHuman_val/Images")
TRAIN_ODGT = os.path.join(BASE_DIR, "annotation_train.odgt")
VAL_ODGT = os.path.join(BASE_DIR, "annotation_val.odgt")

# Crear estructura
for d in [SAVE_DIR, DATA_RAW, TRAIN_IMG_DIR, VAL_IMG_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Usando: {DEVICE} | Modelos en: {SAVE_DIR}")

Usando: cuda | Modelos en: /kaggle/working/models


In [3]:
import os
import glob

# Lista de archivos a descargar
files = ["CrowdHuman_train01.zip", "CrowdHuman_train02.zip", "CrowdHuman_train03.zip",
         "CrowdHuman_val.zip", "annotation_train.odgt", "annotation_val.odgt"]

print("Iniciando descarga desde Hugging Face...")ﬁﬁ
for f in files:
    hf_hub_download(repo_id="sshao0516/CrowdHuman", filename=f, repo_type="dataset", local_dir=DATA_RAW)

# Extracción de imágenes
print("Extrayendo archivos...")
!unzip -q -j "{DATA_RAW}/CrowdHuman_train0*.zip" -d "{TRAIN_IMG_DIR}"

for zip_file in glob.glob(f"{DATA_RAW}/CrowdHuman_train*.zip"):
    os.remove(zip_file)
    
!unzip -q -j "{DATA_RAW}/CrowdHuman_val.zip" -d "{VAL_IMG_DIR}"

!cp -f "{DATA_RAW}/annotation_train.odgt" "{TRAIN_ODGT}"
!cp -f "{DATA_RAW}/annotation_val.odgt" "{VAL_ODGT}"

print(f"Imágenes Train: {len(os.listdir(TRAIN_IMG_DIR))}")
print(f"Imágenes Val: {len(os.listdir(VAL_IMG_DIR))}")

Iniciando descarga desde Hugging Face...


CrowdHuman_train01.zip:   0%|          | 0.00/2.97G [00:00<?, ?B/s]

CrowdHuman_train02.zip:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

CrowdHuman_train03.zip:   0%|          | 0.00/2.31G [00:00<?, ?B/s]

CrowdHuman_val.zip:   0%|          | 0.00/2.49G [00:00<?, ?B/s]

annotation_train.odgt:   0%|          | 0.00/80.0M [00:00<?, ?B/s]

annotation_val.odgt:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

Extrayendo archivos...

3 archives were successfully processed.
Imágenes Train: 15000
Imágenes Val: 4370


In [4]:
class CrowdHumanDataset(Dataset):
    def __init__(self, root, odgt_path, transforms=None):
        self.root = root
        self.transforms = transforms
        with open(odgt_path, 'r') as f:
            self.lines = [json.loads(line) for line in f]

    def __getitem__(self, idx):
        line = self.lines[idx]
        img_path = os.path.join(self.root, f"{line['ID']}.jpg")
        img = Image.open(img_path).convert("RGB")

        boxes, labels = [], []
        for obj in line['gtboxes']:
            if obj['tag'] == 'person' and 'fbox' in obj:
                x, y, w, h = obj['fbox']
                if w > 0 and h > 0:
                    boxes.append([x, y, x + w, y + h])
                    labels.append(1)

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }

        if self.transforms:
            img, target = self.transforms(img, target)
        return img, target

    def __len__(self):
        return len(self.lines)

def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def collate_fn(batch): return tuple(zip(*batch))

In [8]:
# Transformaciones
t_trans = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True), T.RandomHorizontalFlip(0.5)])
v_trans = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

# Dataloaders
train_loader = DataLoader(CrowdHumanDataset(TRAIN_IMG_DIR, TRAIN_ODGT, t_trans),
                          batch_size=2, shuffle=True, num_workers=4, collate_fn=collate_fn)
val_loader = DataLoader(CrowdHumanDataset(VAL_IMG_DIR, VAL_ODGT, v_trans),
                        batch_size=2, shuffle=False, num_workers=4, collate_fn=collate_fn)

model = get_model(num_classes=2).to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.9, weight_decay=0.0005)

In [ ]:
# Listas para almacenar el historial
train_loss_history = []
val_loss_history = []

# Parámetros de Early Stopping
patience = 3
epochs_no_improve = 0
best_val_loss = float('inf')

print("Iniciando entrenamiento...")

for epoch in range(12):
    # FASE DE ENTRENAMIENTO
    model.train()
    epoch_train_loss = 0
    for i, (images, targets) in enumerate(train_loader):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_train_loss += losses.item()

        if i % 100 == 0:
            print(f"E{epoch} I{i}/{len(train_loader)} | Train Loss: {losses.item():.4f}")

    # FASE DE VALIDACIÓN
    epoch_val_loss = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict_val = model(images, targets)
            epoch_val_loss += sum(loss for loss in loss_dict_val.values()).item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    avg_val_loss = epoch_val_loss / len(val_loader)

    # Guardar en el historial
    train_loss_history.append(avg_train_loss)
    val_loss_history.append(avg_val_loss)

    print(f"Resultados Epoca {epoch}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # Lógica de Guardado y Early Stopping
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "last_model.model"))

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.model"))
        print(f"Actualizacion: Mejor modelo guardado con Val Loss: {avg_val_loss:.4f}")
    else:
        epochs_no_improve += 1
        print(f"Sin mejora en validacion. Paciencia: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print(f"Finalizacion por Early Stopping en la epoca {epoch}")
        break

In [ ]:
import matplotlib.pyplot as plt

def plot_losses(train_history, val_history):
    plt.figure(figsize=(10, 6))
    plt.plot(train_history, label='Loss de Entrenamiento', color='blue', linestyle='-')
    plt.plot(val_history, label='Loss de Validacion', color='red', linestyle='-')

    plt.title('Historial de Perdida durante el Entrenamiento')
    plt.xlabel('Epoca')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    # Guardar el grafico en el disco local
    plot_path = os.path.join(BASE_DIR, "curva_entrenamiento.png")
    plt.savefig(plot_path)
    plt.show()
    print(f"Grafico guardado en: {plot_path}")

# Llamar a la funcion
plot_losses(train_loss_history, val_loss_history)

In [13]:
from IPython.display import FileLink
FileLink(r"models/best_model.model")

/kaggle/working/models/best_model.model